# Example to go from MPN cellranger ouput to filtered cells with UMAP and clusters


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import scipy
import seaborn as sns
import celltypist

import pickle
# Load color palette
with open('../data/cell_type_palette.pkl', 'rb') as f:
    cell_type_palette = pickle.load(f)

import sys
sys.path.append('../../1_figure_CL_proof_of_concept/code/')
import utils_00 as gf_utils
large_data_dir = gf_utils.large_data_dir


# Load outputs from cell ranger

In [ ]:
# Parameters
lib = 1
BC = "BC007"


In [ ]:
adata_dir = large_data_dir + 'MPN_WTA/cr_BC007_1_1_sample_filtered_feature_bc_matrix.h5'
adata = sc.read_10x_h5(adata_dir)

adata_dir = large_data_dir + 'MPN_WTA/cr_BC007_1_2_sample_filtered_feature_bc_matrix.h5'
adata2 = sc.read_10x_h5(adata_dir)

adata = adata.concatenate(adata2, batch_key='expt', index_unique='-')

# Filter counts and mito genes

In [ ]:
min_counts = 500
min_genes = 250


In [ ]:
sc.pp.calculate_qc_metrics(adata, inplace = True)

fig = plt.figure(figsize = (8*3, 6*1))
ax = fig.add_subplot(1, 3, 1)
ax.hist(adata.obs['log1p_total_counts'], bins = 100);
ax.set_xlabel('Log-library size', fontsize = 14)
ax.set_ylabel('Frequency', fontsize = 14)
ax.set_title('Histogram of log library size', fontsize = 14)
ax.axvline(np.log1p(min_counts),color='k')

ax = fig.add_subplot(1, 3, 2)
ax.hist(adata.obs['log1p_n_genes_by_counts'], bins = 100);
ax.set_xlabel('Log of num. genes per cell', fontsize = 14)
ax.set_ylabel('Frequency', fontsize = 14)
ax.set_title('Histogram of number of genes expressed in each cell', fontsize = 14)
ax.axvline(np.log1p(min_genes),color='k')

ax = fig.add_subplot(1, 3, 3)
x = adata.obs['log1p_total_counts']
y = adata.obs['log1p_n_genes_by_counts']
ax.scatter(x, y, s = 5);
ax.set_ylabel('Log of num. genes per cell', fontsize = 14)
ax.set_xlabel('Log library size', fontsize = 14)
corr_coef = np.corrcoef(x, y)[0, 1]
ax.set_title('Correlation = ' + str(round(corr_coef, 3)), fontsize = 14)



In [ ]:
# Identify MT-genes
mito_genes = adata.var_names[adata.var_names.str.startswith('MT-')]
print(mito_genes)
index_mito_genes = [adata.var_names.get_loc(j) for j in mito_genes]
mito_frac = np.asarray(np.sum(adata.X[:, index_mito_genes], axis = 1)/np.sum(adata.X, axis = 1)).squeeze() * 100
adata.obs['mito_frac'] = mito_frac
fig = plt.figure(figsize = (8*3, 6*1))
ax = fig.add_subplot(1, 3, 1)
ax.hist(adata.obs['mito_frac'], 100);
ax.set_xlabel('% MT-content', fontsize = 14)
ax.set_ylabel('Frequency', fontsize = 14)

ax = fig.add_subplot(1, 3, 2)
ax.scatter(adata.obs['log1p_total_counts'], adata.obs['mito_frac']);
ax.set_xlabel('Log library size', fontsize = 14)
ax.set_ylabel('% MT-content', fontsize = 14)

ax = fig.add_subplot(1, 3, 3)
ax.scatter(adata.obs['log1p_n_genes_by_counts'], adata.obs['mito_frac']);
ax.set_xlabel('Log num. genes per cell', fontsize = 14)
ax.set_ylabel('% MT-content', fontsize = 14)


In [ ]:
mito_threshold = 10
id_low_mt_cells = adata.obs.index[adata.obs['mito_frac'] < mito_threshold]
adata = adata[id_low_mt_cells, ]


In [ ]:
sc.pp.filter_cells(adata,min_counts=min_counts)
sc.pp.filter_cells(adata,min_genes=min_genes)


In [ ]:
fig = plt.figure(figsize = (8*3, 6*1))
ax = fig.add_subplot(1, 3, 1)
ax.hist(adata.var['n_cells_by_counts'], bins = 100);
ax.set_xlabel('Number of cells a gene is expressed in', fontsize = 14)
ax.set_ylabel('Frequency', fontsize = 14)
ax.set_title('Histogram of number of cells each gene is expressed in', fontsize = 14)

ax = fig.add_subplot(1, 3, 2)
ax.hist(np.log(adata.var['n_cells_by_counts'] + 1), bins = 100);
ax.set_xlabel('Log - Number of cells a gene is expressed in', fontsize = 14)
ax.set_ylabel('Frequency', fontsize = 14)
ax.set_title('Histogram of log of number of cells each gene is expressed in', fontsize = 14)

ax = fig.add_subplot(1, 3, 3)
ax.hist(np.log(adata.var['n_cells_by_counts'] + 1), bins = 100);
ax.set_xlabel('Log - Number of cells a gene is expressed in', fontsize = 14)
ax.set_ylabel('Frequency, ylim trimmed', fontsize = 14)
ax.set_title('Histogram of log of number of cells each gene is expressed in', fontsize = 14)
ax.set_ylim([0, 1000])
(0.0, 1000.0)


# Filter genes, normalize, find HVGs

In [ ]:
sc.pp.filter_genes(adata, min_cells = np.exp(4))
adata.layers['raw_data'] = adata.X.copy()
sc.pp.normalize_total(adata, inplace = True)
adata.layers['norm_counts'] = adata.X.copy()
adata.X = np.log2(adata.X.toarray() + 1)
adata.layers['zs_norm_log'] = scipy.stats.zscore(adata.X)
sc.pp.highly_variable_genes(adata, layer = 'raw_data', n_top_genes = 4000, flavor = 'seurat_v3')
adata.uns['id_hvg'] = np.where(adata.var['highly_variable'])[0]
x = np.log2(adata.var['means'])
y = np.log2(adata.var['variances_norm'])
plt.scatter(x, y, s = 2)
plt.scatter(x[adata.uns['id_hvg']], y[adata.uns['id_hvg']], s = 2, c = 'r')
plt.xlabel('log2 mean expression')
plt.ylabel('log2 normalized variance')
plt.legend(['All genes', 'HVG'])


# Run PCA, clustering, UMAP

In [ ]:
sc.tl.pca(adata, n_comps=100, use_highly_variable=True)
n_PCs = 30
adata.obsm['X_pca'] = adata.obsm['X_pca'][:, 0:n_PCs]
adata.varm['PCs'] = adata.varm['PCs'][:, 0:n_PCs]
adata.uns['loadings'] = adata.varm['PCs'][adata.var['highly_variable'], :]
adata.uns['loadings'] = adata.uns['loadings'][:, 0:n_PCs]

sc.pp.neighbors(adata, n_neighbors=30, use_rep='X_pca', metric='euclidean', key_added='neighbors_30')
sc.tl.umap(adata, neighbors_key = 'neighbors_30', min_dist=0.1)

sc.external.tl.phenograph(adata, clustering_algo='leiden', k=200, jaccard=True, primary_metric='euclidean', 
                          resolution_parameter=1)


In [ ]:
sc.external.tl.phenograph(adata, clustering_algo='leiden', k=200, jaccard=True, primary_metric='euclidean', 
                          resolution_parameter=1)


# Doublet detection and filtering

In [ ]:
adata_scrublet = sc.AnnData(adata.layers['raw_data'], obs = adata.obs, var = adata.var)
sc.external.pp.scrublet(adata_scrublet, sim_doublet_ratio=2.0, expected_doublet_rate=0.06, 
                       knn_dist_metric='euclidean', log_transform=True, n_prin_comps=20, 
                       random_state=0)

adata.obs[['doublet_score', 'predicted_doublet']] = adata_scrublet.obs[['doublet_score', 'predicted_doublet']]
adata.uns['scrublet'] = adata_scrublet.uns['scrublet']


In [ ]:
df_temp = adata.obs[['pheno_leiden', 'predicted_doublet']]

fig = plt.figure(figsize = (12, 4))
ax = fig.add_subplot(1, 1, 1)
sns.barplot(x = 'pheno_leiden', y = 'predicted_doublet', data = df_temp, ax = ax, errorbar = None)

clusters_remove_doublet = df_temp.groupby('pheno_leiden')['predicted_doublet'].mean()[df_temp.groupby('pheno_leiden')['predicted_doublet'].mean() > 0.6].index


In [ ]:
df_temp = adata.obs[['pheno_leiden', 'mito_frac']]
clusters_remove_mito = df_temp.groupby('pheno_leiden')['mito_frac'].mean()[df_temp.groupby('pheno_leiden')['mito_frac'].mean() > 4].index

print(clusters_remove_doublet)
print(clusters_remove_mito)

cells_remove = np.array(np.isin(adata.obs['pheno_leiden'], clusters_remove_doublet) | np.isin(adata.obs['pheno_leiden'], clusters_remove_mito) | (adata.obs['predicted_doublet']) | (adata.obs['mito_frac'] > mito_threshold))
print('removing ' + str(sum(cells_remove)) + ' cells')
adata_clean = sc.AnnData(adata.X[~cells_remove, :], 
                         obs = adata.obs.iloc[~cells_remove, :], 
                         var = adata.var)
adata_clean.layers['norm_counts'] = adata.layers['norm_counts'][~cells_remove, :].copy()
adata_clean.layers['raw_data'] = adata.layers['raw_data'][~cells_remove, :].copy()
adata_clean.layers['zs_norm_log'] = adata.layers['zs_norm_log'][~cells_remove, :].copy()


# Rerun PCA, UMAP, clustering on filtered dataset

In [ ]:
adata = adata_clean.copy()

n_PCs = 30
sc.pp.highly_variable_genes(adata, layer = 'raw_data', n_top_genes = 4000, flavor = 'seurat_v3')
adata.uns['id_hvg'] = np.where(adata.var['highly_variable'])[0]
sc.tl.pca(adata, n_comps=100, use_highly_variable=True)

adata.obsm['X_pca'] = adata.obsm['X_pca'][:, 0:n_PCs]
adata.varm['PCs'] = adata.varm['PCs'][:, 0:n_PCs]
adata.uns['loadings'] = adata.varm['PCs'][adata.var['highly_variable'], :]
adata.uns['loadings'] = adata.uns['loadings'][:, 0:n_PCs]

sc.pp.neighbors(adata, n_neighbors=30, use_rep='X_pca', metric='euclidean', key_added='neighbors_30')
sc.tl.umap(adata, neighbors_key = 'neighbors_30', min_dist=0.1)

sc.external.tl.phenograph(adata, clustering_algo='leiden', k=200, jaccard=True, primary_metric='euclidean', 
                          resolution_parameter=1)

# Annotate cell types

In [ ]:
adata.X = adata.layers['raw_data'].copy()
sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)

sc.pp.highly_variable_genes(adata, layer = 'raw_data', n_top_genes = 4000, flavor = 'seurat_v3')

sc.tl.pca(adata, n_comps=100, use_highly_variable=True)

# CellTypist prediction without over-clustering and majority-voting.
predictions = celltypist.annotate(adata, model = '../data/model_from_MPN8_MPN18.pkl', majority_voting=True)
adata.obs['cell_type'] = predictions.predicted_labels['majority_voting']
adata.X = np.log2(adata.layers['norm_counts'].toarray() + 1)
sc.pl.umap(adata, color = 'cell_type', palette = cell_type_palette)


In [ ]:
### save output as MPN_1_BC007.h5ad
### then add genotype assignments as exemplified in ../../3_figure_FFPE/code/00_probabilistic_genotype_assignment
### MPN feature set defined in 00_define_MPN_feature_set.ipynb